In [1]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)

for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.13.15
Folder: content
numpy - ok
pandas - ok
sklearn - ok


# Build the dataset

In [2]:
import csv
from pathlib import Path
import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)


dataset ready: data/delivery_times.csv


# Loads the data, separates features (X) from the target (y), and does an 80/20 train/test split.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

480 120


# Define a function that trains a model, measures its error on both the training data and the test data, and computes the gap between them.

In [4]:
from sklearn.metrics import mean_absolute_error

def score_both_ways(model, name):
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(name, "train", round(train_mae, 2), "test", round(test_mae, 2), "gap", round(gap, 2))
    return {"name": name, "train": train_mae, "test": test_mae, "gap": gap}

# Model 1: LinearRegression

In [5]:
from sklearn.linear_model import LinearRegression

linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression train 2.04 test 1.92 gap -0.11


# Model 2: decision tree, no limit

In [6]:
from sklearn.tree import DecisionTreeRegressor

wild_tree = score_both_ways(DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train 0.0 test 3.43 gap 3.43


# Model 3: shallow tree (depth 4)

In [7]:
small_tree = score_both_ways(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")

DecisionTree (depth 4) train 3.66 test 4.23 gap 0.57


# Model 4: RandomForest

In [8]:
from sklearn.ensemble import RandomForestRegressor

forest = score_both_ways(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

RandomForest (50 trees) train 1.0 test 2.39 gap 1.4


In [9]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest]).round(2).sort_values("test")
print(results.to_string(index=False))


                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   1.00  2.39  1.40
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57


# Cross-validation - Trains and tests each model 5 times on 5 different data splits and averages the error

In [10]:
from sklearn.model_selection import cross_val_score

def cross_validate(model, name):
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE", round(scores.mean(), 2))
    return scores.mean()

cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")
cv_forest = cross_validate(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LinearRegression MAE 2.03
DecisionTree (depth 4) MAE 4.57
RandomForest (50 trees) MAE 2.69


# Tries several tree depths (2, 3, 4, 6, 8, None), scores each with cross-validation, and finds the best one.

In [11]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=5,
                             scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    scores_by_depth[depth] = round(float(mae), 3)

best_depth = min(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("best depth:", best_depth)

{2: 5.858, 3: 4.85, 4: 4.573, 6: 3.642, 8: 3.395, None: 3.478}
best depth: 8


# Ranks the three model

In [12]:
ranking = sorted(
    {"LinearRegression": cv_linear, "DecisionTree(4)": cv_tree, "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
)
for name, mae in ranking:
    print(name, round(mae, 2))

LinearRegression 2.03
RandomForest(50) 2.69
DecisionTree(4) 4.57


# To Do: Train Logistic Regression, Decision Tree, and Random Forest classifiers on the Breast Cancer dataset, compare their train/test accuracy, use cross-validation to find the best tree depth, and report the confusion matrix and classification metrics to identify the best-performing model. https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset

Step - 1 : Dataset Loading


In [16]:
#Lab 3 - To Do: Breast Cancer Classification
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_cls = pd.DataFrame(data.data, columns=data.feature_names)
y_cls = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

print("Shape:", X_cls.shape)
print("Classes:", dict(zip(data.target_names, [0, 1])))
print(X_cls.head(3))

Shape: (569, 30)
Classes: {np.str_('malignant'): 0, np.str_('benign'): 1}
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38           122.8     1001.0          0.11840   
1        20.57         17.77           132.9     1326.0          0.08474   
2        19.69         21.25           130.0     1203.0          0.10960   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0                 0.07871  ...         25.38          17.33            184.6   
1                 0.05667  ...         24.99          23.41            158.8   
2                 0.05999  ...         23.57          25.53            152.5   


Step 2 — Train/test split (80/20)

In [17]:
from sklearn.model_selection import train_test_split

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

print("Train:", len(Xc_train), " Test:", len(Xc_test))

Train: 455  Test: 114


Step 3 — Helper function


In [18]:
from sklearn.metrics import accuracy_score

def score_classifier(model, name):
    model.fit(Xc_train, yc_train)
    train_acc = accuracy_score(yc_train, model.predict(Xc_train))
    test_acc  = accuracy_score(yc_test,  model.predict(Xc_test))
    gap = test_acc - train_acc
    print(f"{name:35s} train {train_acc:.2f}  test {test_acc:.2f}  gap {gap:+.2f}")
    return {"name": name, "train": train_acc, "test": test_acc, "gap": gap}

Step - 4 : Training Classifiers


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

log_reg    = score_classifier(LogisticRegression(max_iter=5000), "LogisticRegression")
tree_cls   = score_classifier(DecisionTreeClassifier(random_state=42), "DecisionTree (no limit)")
forest_cls = score_classifier(RandomForestClassifier(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LogisticRegression                  train 0.96  test 0.96  gap +0.01
DecisionTree (no limit)             train 1.00  test 0.91  gap -0.09
RandomForest (50 trees)             train 1.00  test 0.96  gap -0.04


Step - 5 : Cross Validation to find better tree depth

In [20]:
from sklearn.model_selection import cross_val_score

depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X_cls, y_cls, cv=5, scoring="accuracy")
    scores_by_depth[depth] = round(float(scores.mean()), 4)

best_depth = max(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("Best depth:", best_depth)

{2: 0.928, 3: 0.9191, 4: 0.9209, 6: 0.9209, 8: 0.9156, None: 0.9173}
Best depth: 2


In [23]:
# Best model — LogisticRegression ( balanced test acc + gap)
best_model = LogisticRegression(max_iter=5000)
best_model.fit(Xc_train, yc_train)
preds = best_model.predict(Xc_test)

from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(yc_test, preds)
print("Confusion matrix:")
print(cm)
print()
print(classification_report(yc_test, preds, target_names=data.target_names))

Confusion matrix:
[[39  3]
 [ 1 71]]

              precision    recall  f1-score   support

   malignant       0.97      0.93      0.95        42
      benign       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [24]:
results_cls = pd.DataFrame([log_reg, tree_cls, forest_cls]).round(3).sort_values("test", ascending=False)
print(results_cls.to_string(index=False))

                   name  train  test    gap
     LogisticRegression  0.956 0.965  0.009
RandomForest (50 trees)  1.000 0.956 -0.044
DecisionTree (no limit)  1.000 0.912 -0.088
